# LAMG+ — run & time it against approximate-Cholesky

[**LAMG+**](https://github.com/orenlivne/lamgplus) is a lean, parameter-free, empirically *O(m)*
algebraic-multigrid solver for graph-Laplacian systems `L x = b`. This notebook shows how to **run
and time** it side-by-side with the two approximate-Cholesky solvers from
[Laplacians.jl](https://github.com/danspielman/Laplacians.jl):

| solver | call | reference |
|---|---|---|
| **LAMG+** | `setup(L)` then `solve(h, b)` | this repo |
| **approxChol** | `approxchol_lap(W)` | Kyng–Sachdeva 2016 (fast) |
| **AC** | `approxchol_lap2(W)` | Gao–Kyng–Spielman 2023 (worst-case robust) |

We run them on (1) a **synthetic grid** and (2) a graph **downloaded from the SuiteSparse test set**,
and report setup/solve time, the **µs/nnz** metric of Gao–Kyng–Spielman, iterations, and the achieved
residual. The two examples land on opposite sides of the paper's **degree crossover**: approximate
Cholesky wins on the low-degree grid; LAMG+ wins on the high-degree finite-element graph.

> **One-time setup** (see [`README`](README.md)): from this directory run
> `julia --project=. -e 'using Pkg; Pkg.instantiate()'`, and make sure you have a Julia Jupyter
> kernel (`julia --project=. -e 'using Pkg; Pkg.add("IJulia")'`).

## Setup
Activate this notebook's environment and load the solvers.

In [1]:
import Pkg
Pkg.activate(@__DIR__; io = devnull)    # examples/ env: LAMG + Laplacians + SuiteSparse loaders

using LAMG, Laplacians, LinearAlgebra, SparseArrays, Random, Printf, UnicodePlots
import LAMG: setup, solve, LAMGOptions, laplacian
using SuiteSparseMatrixCollection                       # fetch graphs from the SuiteSparse collection
include(joinpath(@__DIR__, "mm_reader.jl"))       # read_mm_adj, reduce_to_lcc (shared with examples/)

const TOL   = 1e-8     # relative-residual tolerance every solver is held to
const MAXIT = 300      # iteration / cycle cap
println("ready — Julia ", VERSION)

ready — Julia 1.12.6


## The timing harness

Each solver is held to the same relative residual `‖Lx − b‖ / ‖b‖ ≤ TOL`. We **warm up** once
(Julia JIT-compiles on the first call) and then take the **best of 3** wall-clock runs, split into
*setup* (building the solver / hierarchy) and *solve*. Times are normalized by `nnz(L)` — the
**µs/nnz** metric of Gao–Kyng–Spielman (2023).

LAMG+ consumes the **Laplacian** `L`; `approxchol_lap` / `approxchol_lap2` are Laplacians.jl factories
that consume the **adjacency** `W` and return a solve closure.

In [2]:
relres(L, x, b) = norm(L * x .- b) / max(norm(b), 1e-30)
bestof(f, k = 3) = minimum(@elapsed(f()) for _ in 1:k)

# LAMG+: build the multilevel hierarchy from L, then solve.
function bench_lamg(W, L, b)
    o = LAMGOptions(tol = TOL, max_cycles = MAXIT)
    h = setup(L; options = o); x, info = solve(h, b; options = o)        # warm-up + capture x
    t_set = bestof(() -> setup(L; options = o))
    h2    = setup(L; options = o)
    t_sol = bestof(() -> solve(h2, b; options = o))
    (setup = t_set, solve = t_sol, iters = info.cycles, relres = relres(L, x, b))
end

# approxChol / AC: a factory `make(W) -> f`, where `f(b)` runs preconditioned CG.
function bench_factory(make, W, L, b)
    f0 = make(W; tol = TOL, verbose = false); f0(b)                      # warm-up
    t_set = bestof(() -> make(W; tol = TOL, verbose = false))
    f = make(W; tol = TOL, verbose = false); its = [0]
    x = f(b; tol = TOL, maxits = MAXIT, verbose = false, pcgIts = its)
    t_sol = bestof(() -> f(b; tol = TOL, maxits = MAXIT, verbose = false))
    (setup = t_set, solve = t_sol, iters = its[1], relres = relres(L, x, b))
end

# zero-mean, compatible right-hand side  b = L * x_true
function make_rhs(L; seed = 1)
    n = size(L, 1); rng = MersenneTwister(seed)
    xt = randn(rng, n); xt .-= sum(xt) / n
    L * xt
end

make_rhs (generic function with 1 method)

In [3]:
const SOLVERS = ("LAMG+", "approxChol", "AC")

# Run all three on (W, L), print a table, and return the per-nnz times for plotting.
function run_all(label, W, L; seed = 1)
    n = size(L, 1); nz = nnz(W); b = make_rhs(L; seed = seed)
    @printf("%s\n  n = %d nodes, %d edges, mean degree %.1f\n\n", label, n, nz ÷ 2, nz / n)
    res = Dict(
        "LAMG+"      => bench_lamg(W, L, b),
        "approxChol" => bench_factory(Laplacians.approxchol_lap,  W, L, b),
        "AC"         => bench_factory(Laplacians.approxchol_lap2, W, L, b),
    )
    @printf("%-12s %9s %9s %9s %10s %7s %10s\n", "solver","setup(s)","solve(s)","total(s)","µs/nnz","iters","rel.res")
    println("  ", "-"^70)
    pernnz = Float64[]
    for s in SOLVERS
        r = res[s]; tot = r.setup + r.solve; pn = 1e6 * tot / nz; push!(pernnz, pn)
        @printf("%-12s %9.4f %9.4f %9.4f %10.3f %7d %10.1e\n", s, r.setup, r.solve, tot, pn, r.iters, r.relres)
    end
    best = SOLVERS[argmin(pernnz)]
    @printf("\n  fastest: %s  (%.2f× ahead of the runner-up)\n", best, sort(pernnz)[2] / minimum(pernnz))
    (solvers = collect(SOLVERS), pernnz = pernnz, label = label)
end

plot_pernnz(out) = barplot(out.solvers, out.pernnz;
    title = out.label * "  —  setup+solve µs/nnz (lower is better)", xlabel = "µs/nnz")

plot_pernnz (generic function with 1 method)

## 1. A synthetic graph: the 2-D grid

A `k×k` grid is the canonical structured Laplacian — **low mean degree (≈4)**. Sampled-elimination
solvers (approxChol/AC) are at their best here: small elimination cliques, cheap factor.

In [4]:
# k×k 2-D grid adjacency (5-point stencil), built directly so the example is self-contained.
function grid2d(k)
    idx(i, j) = (j - 1) * k + i
    I = Int[]; J = Int[]
    for j in 1:k, i in 1:k
        i < k && (push!(I, idx(i, j)); push!(J, idx(i + 1, j)))
        j < k && (push!(I, idx(i, j)); push!(J, idx(i, j + 1)))
    end
    A = sparse(I, J, 1.0, k * k, k * k); A + A'
end

W_grid = grid2d(200)                       # 40,000 nodes
L_grid = laplacian(W_grid)
grid_out = run_all("Synthetic 200×200 grid", W_grid, L_grid);

Synthetic 200×200 grid
  n = 40000 nodes, 79600 edges, mean degree 4.0

solver        setup(s)  solve(s)  total(s)     µs/nnz   iters    rel.res


  ----------------------------------------------------------------------
LAMG+           0.0244    0.1014    0.1258      0.790       5    7.1e-10


approxChol      0.0114    0.0182    0.0296      0.186      34    8.9e-09
AC              0.0433    0.0175    0.0608      0.382      21    8.1e-09

  fastest: approxChol  (2.05× ahead of the runner-up)


In [5]:
plot_pernnz(grid_out)

              Synthetic 200×200 grid  —  setup+solve µs/nnz (lower is better) 
              ┌                                        ┐ 
        LAMG+ ┤■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■ 0.79032   
   approxChol ┤■■■■■■■ 0.185983                          
           AC ┤■■■■■■■■■■■■■■■ 0.382059                  
              └                                        ┘ 
                                µs/nnz                   

## 2. A graph from the SuiteSparse test set

`SuiteSparseMatrixCollection.jl` downloads (and caches) any matrix from
[sparse.tamu.edu](https://sparse.tamu.edu). We read it as an **unweighted graph Laplacian** from its
sparsity pattern (via the repo's `mm_reader`) and restrict to the largest connected component.

The default — `Chen/pkustk07`, a 3-D finite-element stiffness matrix — has **high mean degree
(~140)**. This is LAMG+'s regime: its aggregation collapses dense neighbourhoods cheaply while
elimination fills in. Swap in `("SNAP", "web-Stanford")` for a low-degree web graph and watch the
ranking flip.

In [6]:
group, name = "Chen", "pkustk07"          # try ("SNAP","web-Stanford") or ("Boeing","pwtk")
sel  = ssmc_db()
sel  = sel[(sel.group .== group) .& (sel.name .== name), :]
path = joinpath(fetch_ssmc(sel, format = "MM")[1], name * ".mtx")
W_ss, L_ss = reduce_to_lcc(read_mm_adj(path)...)     # pattern -> unweighted Laplacian, largest component
ss_out = run_all("SuiteSparse $(group)/$(name)", W_ss, L_ss);

SuiteSparse Chen/pkustk07
  n = 16860 nodes, 1200972 edges, mean degree 142.5

solver        setup(s)  solve(s)  total(s)     µs/nnz   iters    rel.res


  ----------------------------------------------------------------------
LAMG+           0.1412    0.0368    0.1780      0.074       4    1.5e-09
approxChol      0.3819    0.0442    0.4261      0.177      12    1.9e-09
AC              1.3687    0.0470    1.4157      0.589      10    1.9e-09

  fastest: LAMG+  (2.39× ahead of the runner-up)


In [7]:
plot_pernnz(ss_out)

              SuiteSparse Chen/pkustk07  —  setup+solve µs/nnz (lower is better) 
              ┌                                        ┐ 
        LAMG+ ┤■■■■ 0.0740904                            
   approxChol ┤■■■■■■■■■ 0.177393                        
           AC ┤■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■ 0.589404   
              └                                        ┘ 
                                µs/nnz                   

## Takeaway

Same three solvers, opposite winners:

* **Low-degree grid** → approximate Cholesky wins (cheap sampled elimination).
* **High-degree finite-element graph** → **LAMG+ wins** (lean aggregation; elimination fills in).

That flip is the paper's **degree crossover**: mean degree predicts the winner (LAMG+ when
`d̄ ≳ 30`) at ~91% accuracy across the corpus, because approxChol's fill grows with degree while
LAMG+'s complexity tracks separability instead. To reuse one hierarchy across many right-hand
sides, call `setup(L)` once and `solve(h, b)` per `b`. See the
[paper](../doc/lamg_plus.pdf) and [`examples/`](.) for more.